# 07 — Diagnose before the larger pilot

Timing prediction, agreement with Auger, and true-direction recovery are different endpoints. This notebook investigates the discrepancy without tuning to the reference, and tests an **experimental** signal-based horizontal core estimator. No 1,000-event run is started. Read `docs/prepilot_diagnostics_protocol.md` first. Restart the project kernel and run all cells. Every execution creates a fresh result directory.

The published direction is used **only** in a labelled fixed-direction diagnostic and post-fit reference comparisons. The signal chain uses our plane direction, detector signals, and fixed published core altitude; it does not use published core x/y, direction, energy, or shower-plane distances.

**Coordinate-frame update (2026-10-14).** Reference comparisons now rotate Auger's core-local ENU direction into the central-site ENU frame used by the station coordinates and fitted vectors. No station coordinates or fit objectives are changed. Superseded comparison outputs have been cleared; restart the kernel and run all cells for a new, fully versioned run. Historical result directories remain unchanged. See `docs/conference_coordinate_audit.md`.


In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

from auger_reco.data import direction_reference
from auger_reco.evaluation import prepilot_study

ROOT = Path.cwd().resolve()
while not (ROOT / "src/auger_reco").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Open inside the Auger repository.")
    ROOT = ROOT.parent
study = prepilot_study.create_study(ROOT)
print(f"New run: {study.output}")
print(f"Automatic reference frame: {direction_reference.DIRECTION_FRAME_VERSION}")

## 1. Fix direction; profile time and curvature

For $d_i=r_i-r_c$ and $\rho_i=\|d_i-(d_i\cdot u)u\|$, fit $t_i=t_c+d_i\cdot u/c+A(\rho_i/1000\,\mathrm{m})^2$ at fixed $u$ and core. The remaining two-parameter problem is a weighted linear regression with $A\geq0$. We compare the objective with the free-direction fit, not a naive significance threshold. Saved leave-one-station-out fits reveal influence without automatically rejecting any station.

The fixed published direction is first rotated into the station-coordinate frame. This changes the previously unrotated fixed-direction diagnostic, not the saved unconstrained fits. The shared study configuration records the coordinate-helper source hash and per-event rotation metadata.


In [ ]:
profiles, influence = prepilot_study.run_reference_and_influence(study)
display(profiles.loc[profiles.event_id == prepilot_study.FOCUS_ID])
display(influence.sort_values("maximum_direction_shift_deg", ascending=False))

## 2. Signal-constrained core: explicit experimental assumptions

The softened modified-NKG lateral shape follows the form of Auger Eq. 5.7, but **not its calibrated coefficients or full detector likelihood**. All six fixed slope settings are reported; we do not choose the best agreement with Auger. Core x/y and signal normalization are inferred at our plane direction, with fixed published core z. Only positive valid signal/dsignal with saturation 0 or 1 enter the signal objective. The timing station set is unchanged. Each held-out fold refits the whole plane→signal core→quadratic chain without that station's time or signal. The retained full-event selection and altitude still make this conditional.

In [ ]:
signal_models, signal_folds = prepilot_study.run_signal_experiment(study)
display(signal_models.loc[signal_models.event_id == prepilot_study.FOCUS_ID])

## 3. Known-truth timing and signal controls

Test the measured and flattened geometry at three true directions. Reuse each noise realization across methods and generators. Plane, quadratic, exact expanding sphere, and coherent timing-gradient generators test both matched and mismatched assumptions. A shifted-core fit tests robustness; it is not an estimate of the real core error. Signal controls compare matched and mismatched shapes at two directions. These are toy station-response experiments, not CORSIKA/detector Monte Carlo. The noise-free rows are separate from the 30-trial summaries.

In [4]:
timing_controls, timing_summary = prepilot_study.run_timing_controls(study)
signal_controls = prepilot_study.run_signal_controls(study)
display(timing_summary)
display(
    signal_controls.groupby(["shape", "zenith_true_deg"])[
        ["core_error_m", "angular_error_deg", "known_core_angular_error_deg"]
    ].median()
)

Timing controls: measured_3d, 25.0/20.0, plane


Timing controls: measured_3d, 25.0/20.0, quadratic


Timing controls: measured_3d, 25.0/20.0, sphere


Timing controls: measured_3d, 25.0/20.0, quadratic_coherent_gradient


Timing controls: measured_3d, 54.0/54.0, plane


Timing controls: measured_3d, 54.0/54.0, quadratic


... (16 outputs left out)

Timing controls: flattened, 60.0/210.0, sphere


Timing controls: flattened, 60.0/210.0, quadratic_coherent_gradient


,geometry,zenith_true_deg,azimuth_true_deg,generator,model,attempted,successful,median_error_deg,mean_nominal_reduced_chi_square
0,measured_3d,25.0,20.0,plane,plane,30,30,0.083493,1.009413
1,measured_3d,25.0,20.0,plane,quadratic_true_core,30,30,0.085111,1.041205
2,measured_3d,25.0,20.0,plane,quadratic_shifted_core,30,30,0.087424,1.041205
3,measured_3d,25.0,20.0,quadratic,plane,30,30,0.925506,269.801705
4,measured_3d,25.0,20.0,quadratic,quadratic_true_core,30,30,0.087318,1.016270
...,...,...,...,...,...,...,...,...,...
67,flattened,60.0,210.0,sphere,quadratic_true_core,30,30,0.541948,1.935369
68,flattened,60.0,210.0,sphere,quadratic_shifted_core,30,30,0.500799,1.953347
69,flattened,60.0,210.0,quadratic_coherent_gradient,plane,30,30,2.730181,66.320770
70,flattened,60.0,210.0,quadratic_coherent_gradient,quadratic_true_core,30,30,3.175684,1.386854


core_error_m  angular_error_deg  \
shape      zenith_true_deg                                    
matched    25.0                24.863916           0.105475   
           54.0                19.533520           0.126672   
mismatched 25.0                22.063188           0.107358   
           54.0                24.097205           0.124574   

                            known_core_angular_error_deg  
shape      zenith_true_deg                                
matched    25.0                                 0.090436  
           54.0                                 0.111602  
mismatched 25.0                                 0.090436  
           54.0                                 0.111602

## 4. Saved scientific conclusion and release decision

All adverse outcomes remain visible. Read counts and failure records before summaries. The experimental signal-core estimator is not silently substituted into the pipeline. Figures distinguish reference disagreement, timing quality, and known-truth error.

In [ ]:
conclusion, figures, heldout = prepilot_study.finish_study(
    study,
    profiles,
    influence,
    signal_models,
    signal_folds,
    timing_controls,
    timing_summary,
    signal_controls,
)
display(heldout)
for figure in figures:
    display(Image(filename=str(figure)))
display(Markdown(conclusion))
print(f"Artifacts: {study.output}")